the project will be as following <br>
1 DATA CLEANING <br>
1.1 Loading the dataset and importing the libraries<br>
1.2 missing values <br>
1.3 imputation using data from the names (not best practice)<br>
1.4 dropping missing values to avoid imputation involving categorical data<br>
1.5 binary econcoding categorical data<br>
1.6 gini index<br>
1.7 information gain<br>
2 DATA VIZUALIZATION
2.1 <br>

<h4 align = "center"> Loading the dataset and importing the libraries</h4>


In [253]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import category_encoders as ce
from copy import deepcopy
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_regression
from sklearn.pipeline import Pipeline


file_path = r"D:\ML\dropout_students\video games sales.csv"

df = pd.read_csv(file_path, encoding='ISO-8859-1')

df.head()
df_withnull = deepcopy(df)

In [254]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


<h4 align = "center"> 1.2 Treating the missing values </h4>

since the dataset contains 10K+ entries, the first step will be understanding how much of it is actually useable.<br>


In [255]:
print(df.isnull().sum())

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64


<h4 align = "center"> 1.3 simple imputation using the names to complete the years </h4>

the code below creates a temporary df that will be used to extract the years from the game names that end with a year assuming the game was published $\pm$ 1 year away from the one in the name<br>
the assumption was made because the games that end with a year are mostly sport games, note that games like the more recent "Cyberpunk 2077" could introduce an outlier into the data, resulting is worse models.

In [256]:
null_rows = df[df.isnull().any(axis=1)]

#slices each name and checks is the last 4 characters are digits
null_rows = null_rows[null_rows['Name'].str[-4:].str.isdigit() == True]

with pd.option_context('display.max_rows', None, 'display.max_columns', None): print(null_rows) #allows you to see the whole df without the annoying .tostring formatting

        Rank                               Name Platform    Year       Genre  \
179      180                    Madden NFL 2004      PS2     NaN      Sports   
377      378                   FIFA Soccer 2004      PS2     NaN      Sports   
470      471         wwe Smackdown vs. Raw 2006      PS2     NaN    Fighting   
1649    1651                NASCAR Thunder 2003      PS2     NaN      Racing   
3501    3503                    Madden NFL 2002       XB     NaN      Sports   
4797    4799                   NFL GameDay 2003      PS2     NaN      Sports   
5162    5164                      NBA Live 2003       XB     NaN      Sports   
5669    5671             All-Star Baseball 2005      PS2     NaN      Sports   
5901    5903                      NBA Live 2003       GC     NaN      Sports   
8929    8931             All-Star Baseball 2005       XB     NaN      Sports   
9517    9519             Farming Simulator 2011       PC  2010.0  Simulation   
12922  12924                Tour de Fran

name_dict will store the games names as keys and the approximated years extracted as values, the dict will be used as a temporary way to store the data before being added to the main dataframe.

In [257]:
name_dict = {
    row: row[-4:]
    for row in null_rows['Name']
    if str(row)[-4:].isdigit() and len(str(row)) >= 4
}
print(name_dict)

{'Madden NFL 2004': '2004', 'FIFA Soccer 2004': '2004', 'wwe Smackdown vs. Raw 2006': '2006', 'NASCAR Thunder 2003': '2003', 'Madden NFL 2002': '2002', 'NFL GameDay 2003': '2003', 'NBA Live 2003': '2003', 'All-Star Baseball 2005': '2005', 'Farming Simulator 2011': '2011', 'Tour de France 2011': '2011', 'Sega Rally 2006': '2006', 'Football Manager 2007': '2007', 'PDC World Championship Darts 2008': '2008', 'Driving Simulator 2011': '2011'}


filling the empty data with approximated release years

In [258]:
df['Year'] = df['Year'].fillna(df['Name'].map(name_dict))
print(df.isnull().sum())
print( f"total number of dirty rows is {df.isnull().any(axis=1).sum()}")

Rank              0
Name              0
Platform          0
Year            256
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64
total number of dirty rows is 293


originally there were 271 missing values for the "Year" column, now there are 256, 15 entries were added but the publisher column stayed the same.

In [259]:
count = df.query("Publisher == 'Unknown'").shape[0]
print(count)

203


<h4 align = "center"> 1.4 Treating the rest of the missing values</h4>

Besides the rows that had Nan or none there is data that has "Unknown" as Publisher, wich is another form of missing data <br>
null values will be dropped for the moment, in order to calculate information gain and feauture importance, but will later be treated in the pipeline with sklearn s simpleImputer

In [260]:
df = df.dropna()
print(df.isnull().sum())

Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


<h4 align = "center"> transforming the publishers and platforms with low datapoints </h4><br>
since 2 points is not enough for reliable predictions I chose to stick with 4 and up, everything below 3 datapoints was turned into misc.

In [261]:
print((df['Platform'].value_counts() <= 3).sum())
print((df['Publisher'].value_counts() <= 3).sum())

platform_counts = df['Platform'].value_counts()
publisher_counts = df['Publisher'].value_counts()

df['Platform'] = df['Platform'].map(lambda x: 'Other' if platform_counts[x] <= 3 else x)
df['Publisher'] = df['Publisher'].map(lambda x: 'Unknown' if publisher_counts[x] <= 3 else x)

#quantities
print((df['Platform'] == "Other").sum())
print((df['Publisher'] == "Unknown").sum())

#reassuring the values got transformed
print((df['Platform'].value_counts() <= 3).sum())
print((df['Publisher'].value_counts() <= 3 ).sum())

4
315
7
601
0
0


In [262]:
unknowns = np.array([df.loc[df['Publisher'] == 'Unknown', 'Global_Sales']])
std = unknowns.std()
mean = unknowns.mean()
print(std,mean)

activision = np.array([df.loc[df['Publisher'] == 'Activision', 'Global_Sales']])
std = activision.std()
mean = activision.mean()
print(std,mean)

0.3789859450406492 0.1755574043261231
1.6390636421509786 0.7468012422360248


<h4 align = "center"> 1.5 transforming Year to int</h4>
this step will ensure year is an integer for easier computation

In [263]:
df["Year"] = pd.to_numeric(df["Year"], downcast='integer')
df.info()

<class 'pandas.DataFrame'>
Index: 16305 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16305 non-null  int64  
 1   Name          16305 non-null  str    
 2   Platform      16305 non-null  str    
 3   Year          16305 non-null  int16  
 4   Genre         16305 non-null  str    
 5   Publisher     16305 non-null  str    
 6   NA_Sales      16305 non-null  float64
 7   EU_Sales      16305 non-null  float64
 8   JP_Sales      16305 non-null  float64
 9   Other_Sales   16305 non-null  float64
 10  Global_Sales  16305 non-null  float64
dtypes: float64(5), int16(1), int64(1), str(4)
memory usage: 1.4 MB


<h4 align = "center"> 1.6 categorical data counts</h4>

the next step is to account for the data that is not numerical, since ML models do not do well with categorical data

In [264]:
print(df["Genre"].unique())
print("_"*100)
print(df["Platform"].unique())
print("_"*100)
print(df["Publisher"].unique())


<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy']
Length: 12, dtype: str
____________________________________________________________________________________________________
<StringArray>
[  'Wii',   'NES',    'GB',    'DS',  'X360',   'PS3',   'PS2',  'SNES',
   'GBA',   '3DS',   'PS4',   'N64',    'PS',    'XB',    'PC',  '2600',
   'PSP',  'XOne',    'GC',  'WiiU',   'GEN',    'DC',   'PSV',   'SAT',
   'SCD',    'WS',    'NG', 'Other']
Length: 28, dtype: str
____________________________________________________________________________________________________
<StringArray>
[                   'Nintendo',      'Microsoft Game Studios',
        'Take-Two Interactive', 'Sony Computer Entertainment',
                  'Activision',                     'Ubisoft',
          'Bethesda Softworks',             'Electronic Arts',
       

since there are 31 platforms, 12 Genres and 576 publishers, I am going to use binary encoding<br>
with one hot encoding, the minumum number would be 30+11+575 wich ends up being 616 additional columns, wich is not good.

In [265]:
print(df["Genre"].value_counts())
print(f"there are {len(pd.unique(df['Genre']))} genres")

Genre
Action          3251
Sports          2315
Misc            1686
Role-Playing    1470
Shooter         1282
Adventure       1274
Racing          1228
Platform         875
Simulation       848
Fighting         836
Strategy         670
Puzzle           570
Name: count, dtype: int64
there are 12 genres


In [266]:
print(df["Platform"].value_counts())
print(f"there are {len(pd.unique(df['Platform']))} platforms")

Platform
PS2      2133
DS       2132
PS3      1304
Wii      1290
X360     1236
PSP      1198
PS       1189
PC        938
XB        806
GBA       786
GC        543
3DS       499
PSV       410
PS4       336
N64       316
SNES      239
XOne      213
SAT       173
WiiU      143
2600      116
NES        98
GB         97
DC         52
GEN        27
NG         12
Other       7
SCD         6
WS          6
Name: count, dtype: int64
there are 28 platforms


In [267]:
print(df["Publisher"].value_counts())
print(f"there are {len(pd.unique(df['Publisher']))} publishers")

Publisher
Electronic Arts                 1343
Activision                       966
Namco Bandai Games               928
Ubisoft                          918
Konami Digital Entertainment     823
                                ... 
KID                                4
TGL                                4
Encore                             4
7G//AMES                           4
System Soft                        4
Name: count, Length: 261, dtype: int64
there are 261 publishers


<h4 align = "center"> 1.6 Pearson s correlation </h4>

running a pearson s correlation for numeric data alone is a good way to see if I have any global liniarity,despite having some cases of multicolinearity with values of .9 and up (one even hitting 0.94) caused by the fact that global sales is a sum of the other specific sales. <br>
we can do almost nothing with correlations between various sales
since this is missing all the categorical variables, years do not have a high correlation with any other numerical field, and as stated above we cannot do anything with the corr of different sales we will steer away from any sort of linear model

In [268]:
df.corr(numeric_only=True)

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
Rank,1.000000,0.178261,-0.400496,-0.379277,-0.269142,-0.333082,-0.427202
Year,0.178261,1.000000,-0.091389,0.005976,-0.169324,0.040934,-0.074758
NA_Sales,-0.400496,-0.091389,1.000000,0.768086,0.450881,0.634809,0.941159
EU_Sales,-0.379277,0.005976,0.768086,1.000000,0.436090,0.726141,0.902936
JP_Sales,-0.269142,-0.169324,0.450881,0.436090,1.000000,0.290338,0.612481
Other_Sales,-0.333082,0.040934,0.634809,0.726141,0.290338,1.000000,0.748193
Global_Sales,-0.427202,-0.074758,0.941159,0.902936,0.612481,0.748193,1.000000


<h4 align = "center"> 1.7 binary encoding </h4>

In [272]:
import pandas as pd
import category_encoders as ce

cols_to_encode = ['Name', 'Publisher', 'Platform', 'Genre']

encoder = ce.BinaryEncoder()
df_encoded = encoder.fit_transform(df)
pd.set_option('display.max_columns', None)
df_encoded.head()

,Rank,Name_0,Name_1,Name_2,Name_3,Name_4,Name_5,Name_6,Name_7,Name_8,Name_9,Name_10,Name_11,Name_12,Name_13,Platform_0,Platform_1,Platform_2,Platform_3,Platform_4,Year,Genre_0,Genre_1,Genre_2,Genre_3,Publisher_0,Publisher_1,Publisher_2,Publisher_3,Publisher_4,Publisher_5,Publisher_6,Publisher_7,Publisher_8,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,2006,0,0,0,1,0,0,0,0,0,0,0,0,1,41.49,29.02,3.77,8.46,82.74
1,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1985,0,0,1,0,0,0,0,0,0,0,0,0,1,29.08,3.58,6.81,0.77,40.24
2,3,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,2008,0,0,1,1,0,0,0,0,0,0,0,0,1,15.85,12.88,3.79,3.31,35.82
3,4,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,2009,0,0,0,1,0,0,0,0,0,0,0,0,1,15.75,11.01,3.28,2.96,33.00
4,5,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,1,1996,0,1,0,0,0,0,0,0,0,0,0,0,1,11.27,8.89,10.22,1.00,31.37


In [ ]:
df.head()

<h4 align = "center"> 1.9 information gain </h4>

we will first use information gain in order to know if there is a relationship between eace feauture and the target, it is based on "entropy estimation from k-nearest neighbors distances" (sklearn mutual_info_regression documentation), this will help decide the feautures used in the actual model training and testing

after doing a mutual information, we find out publisher is the only one with a moderate relationship. this show us that the publisher is at least twice more informative than the platform. This could show us that brand loyalty and good previous results will improve the probabily of earning bigger sums than the genre or the platform the game is on

<h4 align = "center"> 1.10 feauture importance
 </h4>

this is a mean increase in impurity feauture importance calculation, it is defined as "the mean and standard deviation of accumulation of the impurity decrease within each tree." ( sklearn Feature importances with a forest of trees), this basically shows how much each feautures reduces entropy.<br>
as stated on sklearn " Impurity-based feature importances can be misleading for high cardinality ", since in this df publisher is the one with the highest cardinality, and it has less than 5%, we will not go the extra step for feauture permutation


In [ ]:
from sklearn.tree import DecisionTreeRegressor


model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(x, y)

combined_importance = pd.Series(model.feature_importances_, index=x.columns).sort_values(ascending=True)

print(combined_importance)
combined_importance.plot.bar()

we find that the name and genre are way more relevant when testing feauture importance than the publisher meaning the model found a relationship between genre and name

<h4 align = "center"> 2 preparing the data pipeline </h4>

<h4 align = "center"> 2 separating target and feautures + numerical and categorical data </h4>
notice that rank might tell the model information about the targets, for this reason we will be removing the rank

In [ ]:
df_withnull["Year"] = pd.to_numeric(df_withnull["Year"], downcast='integer') # stays float because of nulls
df_withnull = df_withnull.drop(["Rank"], axis =  1)
df_withnull.info()

we will be splitting the df into targets Y and feautures X <br>
we will also split numerical and categorical data in order to prepare the feautures independently

In [ ]:
X = df_withnull.drop(columns = ["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"])
Y = df_withnull[["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"]]

numerical = ["Year"]
categorical = ['Name','Platform','Publisher','Genre']


for numerical values, in this case year alone, simple imputer will be used, it will "Replace missing values using a descriptive statistic","If “most_frequent”, then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value" (sklearn SimpleImputer) <br>
the reason I am using 2 pipelines is due to the fact that I want to account for year too.

In [ ]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])
categorical_pipeline = Pipeline([
    # add imputer and encoder imputer will fill publisher to indie indie being publishers with 1 single game
])

In [ ]:
#scatterplot of sales per year, we are trying to see any linear trends in our data in order to decide if we want to use a linear model or not
plt.figure()
sns.scatterplot(data = df, x="Year", y="Global_Sales")
plt.show()


In [ ]:
#using regplot you can get a general ideea of how a line will fit to you data, in this case the line is almost straight and we can observe heteroscedasticity, the spread is not consistent, from 2005- 2016 the has been a rise in sales, but the line is not tilted upwords in our case since there are many datapoints close to 0

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Global_Sales'] = pd.to_numeric(df['Global_Sales'], errors='coerce')


plt.figure(figsize=(10, 6))
sns.regplot(data=df, x="Year", y="Global_Sales",line_kws={"color": "red"})
plt.show()